In [1]:
from google.colab import files

uploaded = files.upload()

Saving energy_master.csv to energy_master.csv


In [5]:
import pandas as pd

energy = pd.read_csv('energy_master.csv')
energy['utc_timestamp'] = pd.to_datetime(energy['utc_timestamp'])
energy = energy.set_index('utc_timestamp')

rl = energy[['price']].copy()

rl['hour'] = rl.index.hour

print(rl.shape)
rl.head()

(17534, 2)


,price,hour
utc_timestamp,,
2018-10-01 05:00:00+00:00,77.32,5
2018-10-01 06:00:00+00:00,84.97,6
2018-10-01 07:00:00+00:00,79.56,7
2018-10-01 08:00:00+00:00,73.70,8
2018-10-01 09:00:00+00:00,71.63,9


In [6]:
#split prices in 10 categories
rl['price_bin'] = pd.qcut(
    rl['price'],
    q=10,
    labels=False
)

print(
    rl['price_bin']
    .value_counts()
    .sort_index()
)

price_bin
0    1755
1    1752
2    1754
3    1756
4    1754
5    1752
6    1751
7    1754
8    1752
9    1754
Name: count, dtype: int64


In [7]:
#Battery State
battery_states = [0,25,50,75,100]

actions = [
    'hold',
    'charge',
    'discharge'
]

print(battery_states)
print(actions)

[0, 25, 50, 75, 100]
['hold', 'charge', 'discharge']


In [8]:
#Q-table
import numpy as np

battery_states = [0,25,50,75,100]
actions = ['hold','charge','discharge']

Q = np.zeros(
    (
        10,                       # price bins
        len(battery_states),      # battery levels
        24,                       # hours
        len(actions)              # actions
    )
)

print(Q.shape)

(10, 5, 24, 3)


In [9]:
#Hyperparameters
alpha = 0.10      # learning rate
gamma = 0.95      # future reward
epsilon = 0.20    # exploration

capacity = 100
rate = 25

print(alpha, gamma, epsilon)

0.1 0.95 0.2


In [10]:
#train
battery = 50
profit = 0

for i in range(len(rl)-1):

    price_bin = int(rl.iloc[i]['price_bin'])
    hour = int(rl.iloc[i]['hour'])

    battery_idx = battery_states.index(battery)

    # epsilon-greedy
    if np.random.rand() < epsilon:
        action = np.random.randint(3)
    else:
        action = np.argmax(
            Q[
                price_bin,
                battery_idx,
                hour
            ]
        )

    price = rl.iloc[i]['price']

    reward = 0
    new_battery = battery

    # charge
    if action == 1 and battery < capacity:
        new_battery = min(
            capacity,
            battery + rate
        )
        reward = -price * rate

    # discharge
    elif action == 2 and battery > 0:
        new_battery = max(
            0,
            battery - rate
        )
        reward = price * rate

    profit += reward

    next_bin = int(
        rl.iloc[i+1]['price_bin']
    )

    next_hour = int(
        rl.iloc[i+1]['hour']
    )

    next_idx = battery_states.index(
        new_battery
    )

    Q[
        price_bin,
        battery_idx,
        hour,
        action
    ] += alpha * (
        reward
        + gamma * np.max(
            Q[
                next_bin,
                next_idx,
                next_hour
            ]
        )
        - Q[
            price_bin,
            battery_idx,
            hour,
            action
        ]
    )

    battery = new_battery

print("Episode profit:", profit)
print("Final battery:", battery)

Episode profit: 81944.0
Final battery: 0


Δεν μας ενδιαφέρει ακόμα το profit.

Θέλουμε να ελέγξουμε ότι:

το Q-table ενημερώνεται,
ο agent μπορεί να κάνει charge/discharge,
δεν εμφανίζονται σφάλματα.

Ο agent:

✅ κατάλαβε τις καταστάσεις (states)

✅ έκανε charge/discharge

✅ ενημέρωσε το Q-table

✅ παρήγαγε θετικό profit

In [11]:
#training loop
episodes = 100

profits = []

for ep in range(episodes):

    battery = 50
    episode_profit = 0

    for i in range(len(rl)-1):

        price_bin = int(rl.iloc[i]['price_bin'])
        hour = int(rl.iloc[i]['hour'])

        battery_idx = battery_states.index(battery)

        # exploration
        if np.random.rand() < epsilon:
            action = np.random.randint(3)
        else:
            action = np.argmax(
                Q[price_bin,
                  battery_idx,
                  hour]
            )

        price = rl.iloc[i]['price']

        reward = 0
        new_battery = battery

        # charge
        if action == 1 and battery < capacity:
            new_battery = min(
                capacity,
                battery + rate
            )
            reward = -price*rate

        # discharge
        elif action == 2 and battery > 0:
            new_battery = max(
                0,
                battery-rate
            )
            reward = price*rate

        episode_profit += reward

        next_bin = int(
            rl.iloc[i+1]['price_bin']
        )

        next_hour = int(
            rl.iloc[i+1]['hour']
        )

        next_idx = battery_states.index(
            new_battery
        )

        Q[
            price_bin,
            battery_idx,
            hour,
            action
        ] += alpha*(
            reward
            + gamma*np.max(
                Q[
                    next_bin,
                    next_idx,
                    next_hour
                ]
            )
            - Q[
                price_bin,
                battery_idx,
                hour,
                action
            ]
        )

        battery = new_battery

    profits.append(episode_profit)

    if ep % 10 == 0:
        print(ep, episode_profit)

0 110336.5
10 346146.75
20 451209.25
30 584802.5
40 661792.75
50 717456.0
60 757312.75
70 806860.75
80 853981.0
90 907362.5


In [12]:
print()
print("Best profit:", max(profits))
print("Last profit:", profits[-1])
print("Average:", np.mean(profits[-10:]))


Best profit: 953769.25
Last profit: 953769.25
Average: 922120.9


agent ουσιαστικά ανακάλυψε:

charge → low price regimes
discharge → high price regimes